In [1]:
import pandas as pd
import scanpy as sc
import seaborn as sns
import numpy as np

In [2]:
# Load the expression matrix
expr=pd.read_csv('GSE153855_Expression_counts_HQ_allsamples.txt', sep='\t', index_col=0, header=None)
# And annotations data
ann=pd.read_csv('GSE153855_Cell_annotation.txt', sep='\t', index_col=0)
ann=ann.reset_index()
ann.index=[str(i+1) for i in ann.index.values]

In [3]:
# Generate AnnData object
adata=sc.AnnData(expr.T)
adata.obs=adata.obs.merge(ann, left_index=True, right_index=True)

C:\Users\Leo\anaconda3\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [4]:
# Select beta cells
beta_cells=adata[adata.obs.CellType=='Beta'].copy()
beta_cells.var.columns = beta_cells.var.columns.astype(str)
beta_cells.obs.columns = beta_cells.obs.columns.astype(str)
beta_cells.var.index.name='gene_name'
beta_cells.write_loom('Dataset1_betas_raw.loom')

## Pseudo-bulk



In [5]:
# Makge pseudo-counts
counts_df=pd.DataFrame(beta_cells.X)
counts_df.columns=adata.var_names
counts_df['Donor']=beta_cells.obs.Donor.values
pseudo=counts_df.groupby('Donor').sum()

In [6]:
# Make donor-level annotations
donor_ann=ann.drop_duplicates(subset='Donor')
donor_ann.index=donor_ann.Donor.values
cond_=donor_ann.loc[pseudo.index][['Disease']]

In [7]:
adata_pseudo=sc.AnnData(pseudo)
adata_pseudo.obs=adata_pseudo.obs.merge(donor_ann, left_index=True, right_index=True)

In [8]:
adata_pseudo

AnnData object with n_obs × n_vars = 11 × 30558
    obs: 'Donor', 'Disease', 'CellType'

In [9]:
adata_pseudo.var=adata_pseudo.var.reset_index()
adata_pseudo.var.columns=['gene_name']
adata_pseudo.var.index=adata_pseudo.var.gene_name

C:\Users\Leo\anaconda3\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [10]:
adata_pseudo.write_loom('Dataset1_Pseudo_betas.loom')